# POI Recommendation System for Vietnam Destinations

This notebook implements the full POI recommendation workflow for `data/vietnam_destinations_google_maps_browser_hotosm.xlsx`:

1. Inspect the scraped destination data.
2. Clean invalid/missing rows.
3. Fill missing values for recommendation-safe fields.
4. Normalize ratings, review counts, distance, text, and keyword features.
5. Build separated criteria scores for content, type, location, distance, and quality.
6. Use **Fuzzy AHP** as the main ranking model.
7. Evaluate the main model against baseline models with ranking metrics.
8. Save cleaned data, sample recommendations, and evaluation results.

Important modeling decision: the pairwise comparison matrix is not generated from preset weights. It is an explicit input representing user/expert judgments. AHP computes the criteria weights from that matrix and checks consistency.

In [ ]:
from pathlib import Path
import math
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from rapidfuzz import fuzz
    RAPIDFUZZ_AVAILABLE = True
except ImportError:
    from difflib import SequenceMatcher
    RAPIDFUZZ_AVAILABLE = False

    class _FallbackFuzz:
        @staticmethod
        def ratio(a, b):
            return int(100 * SequenceMatcher(None, str(a), str(b)).ratio())

        @staticmethod
        def partial_ratio(a, b):
            a = str(a)
            b = str(b)
            if not a or not b:
                return 0
            shorter, longer = (a, b) if len(a) <= len(b) else (b, a)
            if shorter and shorter in longer:
                return 100
            return _FallbackFuzz.ratio(a, b)

        @staticmethod
        def token_set_ratio(a, b):
            a_tokens = set(str(a).split())
            b_tokens = set(str(b).split())
            if not a_tokens or not b_tokens:
                return 0
            shared = a_tokens & b_tokens
            union = a_tokens | b_tokens
            return int(100 * len(shared) / len(union))

    fuzz = _FallbackFuzz()

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)

DATA_PATH = Path('data/vietnam_destinations_google_maps_browser_hotosm.xlsx')
CLEAN_OUTPUT_PATH = Path('data/poi_recommendation_cleaned.xlsx')
SAMPLE_RECOMMENDATIONS_PATH = Path('data/poi_sample_recommendations.xlsx')
EVALUATION_OUTPUT_PATH = Path('data/poi_evaluation_metrics.xlsx')

print('RapidFuzz available:', RAPIDFUZZ_AVAILABLE)
print('Input exists:', DATA_PATH.exists(), DATA_PATH)

## 1. Load and Inspect Data

Always inspect the scraped workbook before modeling. This identifies missing columns, null-heavy fields, weird data types, coordinate problems, and fields that should be parsed before recommendation.

In [ ]:
df_raw = pd.read_excel(DATA_PATH)

print('Shape:', df_raw.shape)
print('Columns:')
for idx, col in enumerate(df_raw.columns, start=1):
    print(f'{idx:02d}. {col}')

print('\nDtypes:')
display(df_raw.dtypes.to_frame('dtype'))

print('\nMissing values:')
display(df_raw.isna().sum().sort_values(ascending=False).to_frame('missing_count'))

print('\nSample rows:')
display(df_raw.head(5))
display(df_raw.tail(5))

In [ ]:
# Quick uniqueness and coordinate sanity checks.
inspect_df = df_raw.copy()
inspect_df['maps_latitude_num'] = pd.to_numeric(inspect_df.get('maps_latitude'), errors='coerce')
inspect_df['maps_longitude_num'] = pd.to_numeric(inspect_df.get('maps_longitude'), errors='coerce')

summary = {
    'rows': len(inspect_df),
    'unique_names': inspect_df['Tên địa điểm'].astype(str).str.strip().nunique(),
    'duplicate_name_rows': inspect_df.duplicated(subset=['Tên địa điểm']).sum(),
    'missing_latitude': inspect_df['maps_latitude_num'].isna().sum(),
    'missing_longitude': inspect_df['maps_longitude_num'].isna().sum(),
    'lat_outside_vietnam_bbox': (~inspect_df['maps_latitude_num'].between(7.0, 24.5)).sum(),
    'lng_outside_vietnam_bbox': (~inspect_df['maps_longitude_num'].between(102.0, 110.5)).sum(),
}
display(pd.Series(summary, name='value').to_frame())

print('Top destination types:')
display(df_raw['maps_destination_type'].fillna('Unknown').value_counts().head(20).to_frame('count'))

## 2. Helper Functions

These functions standardize Vietnamese text, parse ratings/reviews, clean keywords/hours, calculate distance, and define fuzzy membership curves.

In [ ]:
def strip_accents(value):
    text = unicodedata.normalize('NFD', str(value or ''))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('\u0111', 'd').replace('\u0110', 'D')


def normalize_text(value):
    text = strip_accents(value).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return ' '.join(text.split())


def clean_string(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else np.nan


def parse_rating(value):
    if pd.isna(value):
        return np.nan
    text = str(value).lower().replace(',', '.')
    match = re.search(r'(\d+(?:\.\d+)?)', text)
    if not match:
        return np.nan
    rating = float(match.group(1))
    if 0 <= rating <= 5:
        return rating
    if 0 <= rating <= 10:
        return rating / 2
    return np.nan


def parse_review_count(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ')
    match = re.search(r'([\d][\d,\.\s]*)', text)
    if not match:
        return np.nan
    digits = re.sub(r'\D', '', match.group(1))
    return float(digits) if digits else np.nan


def split_keywords(value):
    if pd.isna(value):
        return []
    text = str(value).replace('\u201c', '"').replace('\u201d', '"')
    parts = re.split(r'[,;|]', text)
    cleaned = []
    for part in parts:
        token = part.strip().strip('"').strip("'").strip()
        token = re.sub(r'\s+', ' ', token)
        if token:
            cleaned.append(token)
    return list(dict.fromkeys(cleaned))


def keywords_to_style(values):
    values = [str(v).strip() for v in values if str(v).strip()]
    values = list(dict.fromkeys(values))
    return ' , '.join(f'"{v}"' for v in values)


def infer_keywords(row):
    keywords = split_keywords(row.get('Từ Khóa'))
    type_text = normalize_text(row.get('maps_destination_type'))
    desc_text = normalize_text(row.get('Mô tả'))
    name_text = normalize_text(row.get('Tên địa điểm'))
    joined = ' '.join([type_text, desc_text, name_text])

    rules = [
        (['beach', 'bien', 'bai tam'], ['biển', 'nghỉ dưỡng', 'chụp ảnh']),
        (['museum', 'bao tang'], ['bảo tàng', 'văn hóa', 'tham quan']),
        (['temple', 'pagoda', 'chua', 'den', 'dinh', 'place worship'], ['tâm linh', 'văn hóa', 'tham quan']),
        (['viewpoint', 'ngam canh'], ['ngắm cảnh', 'chụp ảnh', 'tham quan']),
        (['theme park', 'amusement', 'zoo', 'aquarium'], ['vui chơi', 'giải trí', 'tham quan']),
        (['historic', 'heritage', 'di tich', 'thanh co'], ['di tích', 'lịch sử', 'tham quan']),
        (['park', 'garden', 'camp'], ['dã ngoại', 'vui chơi', 'tham quan']),
        (['market', 'cho'], ['chợ', 'văn hóa', 'trải nghiệm']),
    ]
    for needles, additions in rules:
        if any(needle in joined for needle in needles):
            keywords.extend(additions)

    if not keywords:
        keywords = ['tham quan', 'du lịch']
    return list(dict.fromkeys(keywords))


def clean_hours(value):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ')
    # Remove Google UI icons / repeated separators. Keep simple opening-hour strings.
    chunks = [c.strip() for c in text.split('|')]
    chunks = [c for c in chunks if c and not re.fullmatch(r'[\W_]+', c)]
    chunks = [c for c in chunks if 'busy at' not in c.lower()]
    if not chunks:
        return np.nan
    return ' | '.join(list(dict.fromkeys(chunks)))


def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in [lat1, lon1, lat2, lon2]):
        return np.nan
    radius = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return 2 * radius * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def minmax(series):
    series = pd.to_numeric(series, errors='coerce')
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value) or min_value == max_value:
        return pd.Series(0.0, index=series.index)
    return (series - min_value) / (max_value - min_value)


def ramp_down(x, low, high):
    if pd.isna(x):
        return 0.0
    if x <= low:
        return 1.0
    if x >= high:
        return 0.0
    return float((high - x) / (high - low))


def fuzzy_text_score(query, text):
    query_norm = normalize_text(query)
    text_norm = normalize_text(text)
    if not query_norm or not text_norm:
        return 0.0
    return max(
        fuzz.ratio(query_norm, text_norm),
        fuzz.partial_ratio(query_norm, text_norm),
        fuzz.token_set_ratio(query_norm, text_norm),
    ) / 100

## 3. Clean Missing Rows and Fill Missing Data

Essential fields for POI recommendation are name, latitude, longitude, text/category, rating, and review count. Rows missing name or valid Vietnam coordinates are removed. Other missing fields are filled with recommendation-safe defaults.

In [ ]:
df = df_raw.copy()

# Standardize text columns.
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].map(clean_string)

# Parse numeric fields.
df['rating'] = df['Đánh giá '].map(parse_rating)
df['maps_review_count_clean'] = pd.to_numeric(df['maps_review_count'], errors='coerce')
missing_review_mask = df['maps_review_count_clean'].isna()
df.loc[missing_review_mask, 'maps_review_count_clean'] = df.loc[missing_review_mask, 'maps_review_label'].map(parse_review_count)

df['latitude'] = pd.to_numeric(df['maps_latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['maps_longitude'], errors='coerce')

# Drop rows that cannot be recommended geographically.
before_rows = len(df)
df = df.dropna(subset=['Tên địa điểm', 'latitude', 'longitude']).copy()
df = df[df['latitude'].between(7.0, 24.5) & df['longitude'].between(102.0, 110.5)].copy()
print('Dropped rows with missing/invalid core fields:', before_rows - len(df))

# Deduplicate near-identical POIs.
df['name_key'] = df['Tên địa điểm'].map(normalize_text)
df['coord_key'] = df['latitude'].round(5).astype(str) + ',' + df['longitude'].round(5).astype(str)
before_dedup = len(df)
df = df.drop_duplicates(subset=['name_key', 'coord_key']).copy()
print('Dropped duplicated name+coordinate rows:', before_dedup - len(df))

# Fill categorical/text fields.
df['poi_type'] = df['maps_destination_type'].fillna('Unknown').astype(str).str.strip()
df['result_name'] = df['maps_result_name'].fillna(df['Tên địa điểm'])
df['location'] = df['Vị trí'].fillna('Việt Nam')
df['description'] = df['Mô tả'].fillna('Điểm đến tại Việt Nam.')
df['image_url'] = df['Ảnh'].fillna('')
df['open_hours_clean'] = df['maps_open_hours'].fillna(df['maps_first_open_hours']).map(clean_hours)
df['open_hours_clean'] = df['open_hours_clean'].fillna('Không rõ')

# Fill keywords with same quoted style as the source workbook.
df['keyword_list'] = df.apply(infer_keywords, axis=1)
df['keywords_clean'] = df['keyword_list'].map(keywords_to_style)

# Fill ratings by type median, then global median.
global_rating_median = df['rating'].median()
type_rating_median = df.groupby('poi_type')['rating'].transform('median')
df['rating_filled'] = df['rating'].fillna(type_rating_median).fillna(global_rating_median).fillna(3.5)
df['rating_filled'] = df['rating_filled'].clip(0, 5)

# Review counts are popularity signals; missing reviews become 0 rather than imputed popularity.
df['review_count'] = df['maps_review_count_clean'].fillna(0).clip(lower=0)
df['has_image'] = (df['image_url'].astype(str).str.len() > 0).astype(int)
df['has_hours'] = (df['open_hours_clean'] != 'Không rõ').astype(int)

# Split criteria text to avoid double-counting.
# Content excludes type and location. Type/location are scored by separate criteria.
df['content_text'] = (
    df['Tên địa điểm'].fillna('') + ' ' +
    df['result_name'].fillna('') + ' ' +
    df['description'].fillna('') + ' ' +
    df['keyword_list'].map(lambda values: ' '.join(values))
)
df['content_text_norm'] = df['content_text'].map(normalize_text)

# Search text is only for direct fuzzy lookup, not for the content criterion.
df['search_text'] = (
    df['content_text'].fillna('') + ' ' +
    df['location'].fillna('') + ' ' +
    df['poi_type'].fillna('')
)
df['search_text_norm'] = df['search_text'].map(normalize_text)

# Keep source-style columns updated for cleaned export.
df['Đánh giá '] = df['rating_filled'].round(2)
df['Từ Khóa'] = df['keywords_clean']
df['maps_latitude'] = df['latitude']
df['maps_longitude'] = df['longitude']

print('Cleaned shape:', df.shape)
display(df[['Tên địa điểm', 'location', 'poi_type', 'rating', 'rating_filled', 'review_count', 'keywords_clean']].head(10))

## 4. Normalize Features

Normalization brings different criteria onto comparable `[0, 1]` scales: rating, reviews, image availability, hours availability, and text vectors.

In [ ]:
df['rating_norm'] = (df['rating_filled'] / 5).clip(0, 1)
df['review_log'] = np.log1p(df['review_count'])
df['review_norm'] = minmax(df['review_log'])
df['quality_score'] = (
    0.55 * df['rating_norm'] +
    0.35 * df['review_norm'] +
    0.05 * df['has_image'] +
    0.05 * df['has_hours']
).clip(0, 1)

clean_report = pd.DataFrame({
    'column': ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours'],
    'min': [df[c].min() for c in ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours']],
    'max': [df[c].max() for c in ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours']],
    'mean': [df[c].mean() for c in ['rating_norm', 'review_norm', 'quality_score', 'has_image', 'has_hours']],
})
display(clean_report)

print('Missing after preprocessing:')
display(df[['Tên địa điểm', 'location', 'description', 'keywords_clean', 'latitude', 'longitude', 'rating_filled', 'review_count']].isna().sum().to_frame('missing_count'))

## 5. Build Content Text Features

TF-IDF is built only from content fields: destination name, result name, description, and keywords. Type and location are intentionally excluded because they are separate Fuzzy AHP criteria.

In [ ]:
content_tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
)
content_matrix = content_tfidf.fit_transform(df['content_text_norm'])

print('Content TF-IDF matrix shape:', content_matrix.shape)
print('Content criterion excludes `poi_type` and `location` to avoid double-counting.')

feature_preview = pd.DataFrame({
    'feature': content_tfidf.get_feature_names_out()[:30]
})
display(feature_preview.T)

## 6. Criteria Scoring for Fuzzy AHP

The ranking model uses five separated criteria:

- `content`: semantic/text match using only name, description, and keywords
- `type`: fuzzy match against `maps_destination_type`
- `location`: fuzzy match against `V? tr?`
- `distance`: Haversine distance from user coordinates
- `quality`: rating, reviews, image availability, and opening-hour availability

Content intentionally excludes type and location. This prevents the same signal from being counted twice.

In [ ]:
def content_similarity_scores(profile):
    query_parts = []
    profile = profile or {}
    for key in ['query', 'keywords']:
        value = profile.get(key, [])
        if isinstance(value, str):
            query_parts.append(value)
        else:
            query_parts.extend([str(item) for item in value])
    query_text = normalize_text(' '.join(query_parts))
    if not query_text:
        return np.zeros(len(df))
    query_vector = content_tfidf.transform([query_text])
    return cosine_similarity(query_vector, content_matrix).ravel()


def fuzzy_list_similarity(values, text_series):
    if isinstance(values, str):
        values = [values]
    values = [value for value in (values or []) if str(value).strip()]
    if not values:
        return np.zeros(len(text_series))

    scores = []
    for text in text_series.fillna('').astype(str):
        text_score = max(fuzzy_text_score(value, text) for value in values)
        scores.append(text_score)
    return np.array(scores)


def distance_scores(profile):
    lat = profile.get('current_lat') if profile else None
    lng = profile.get('current_lng') if profile else None
    max_distance_km = profile.get('max_distance_km', 300) if profile else 300
    if lat is None or lng is None:
        return np.ones(len(df)), np.full(len(df), np.nan)

    distances = np.array([
        haversine_km(lat, lng, row_lat, row_lng)
        for row_lat, row_lng in zip(df['latitude'], df['longitude'])
    ])
    scores = np.array([ramp_down(distance, 0, max_distance_km) for distance in distances])
    return scores, distances


def criteria_scores_for_profile(profile):
    profile = profile or {}
    content_scores = content_similarity_scores(profile)
    type_scores = fuzzy_list_similarity(profile.get('preferred_types'), df['poi_type'])
    location_scores = fuzzy_list_similarity(profile.get('preferred_locations'), df['location'])
    distance_score_values, distances = distance_scores(profile)
    quality = df['quality_score'].to_numpy()
    return pd.DataFrame({
        'content_criterion': content_scores,
        'type_criterion': type_scores,
        'location_criterion': location_scores,
        'distance_criterion': distance_score_values,
        'quality_criterion': quality,
        'distance_km': distances,
    }, index=df.index)

## 7. Main Model: Fuzzy AHP Ranking

This is the main ranking model. It follows the Fuzzy AHP knowledge from `docs/fuzzy_ahp.md`:

```text
Hierarchy -> pairwise comparison -> AHP weights + CR -> fuzzy weights
-> fuzzy evaluation matrix -> aggregation H = A x W -> defuzzification
-> normalization -> ranking
```

The pairwise matrix below is an **input** from user/expert assessment. It is not generated from preset weights. Edit `EXPERT_PAIRWISE_MATRIX` if another expert gives different judgments.

The final ranking score is:

```text
final_score = fuzzy_ahp_norm
```

where `fuzzy_ahp_norm` is the normalized defuzzified Fuzzy AHP score.

In [ ]:
FUZZY_AHP_CRITERIA = ['content', 'type', 'location', 'distance', 'quality']

# Expert/user pairwise comparison input using Saaty's scale.
# Criteria order: content, type, location, distance, quality.
# Example interpretation:
# - content is moderately more important than location: 3
# - quality is equally important to content: 1
# - type and distance are equally important: 1
EXPERT_PAIRWISE_MATRIX = np.array([
    [1,   2,   3,   2,   1],
    [1/2, 1,   2,   1,   1/2],
    [1/3, 1/2, 1,   1/2, 1/3],
    [1/2, 1,   2,   1,   1/2],
    [1,   2,   3,   2,   1],
], dtype=float)

FUZZY_AHP_INFO = {}
RI_TABLE = {
    1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12,
    6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49,
}


def ahp_weights_from_pairwise(pairwise):
    matrix = np.asarray(pairwise, dtype=float)
    col_sum = matrix.sum(axis=0)
    normalized = matrix / col_sum
    weights = normalized.mean(axis=1)
    weights = weights / weights.sum()

    n = matrix.shape[0]
    weighted_sum = matrix @ weights
    consistency_vector = weighted_sum / weights
    lambda_max = consistency_vector.mean()
    ci = (lambda_max - n) / (n - 1) if n > 1 else 0.0
    ri = RI_TABLE.get(n, RI_TABLE[max(RI_TABLE)])
    cr = 0.0 if ri == 0 else ci / ri
    return weights, lambda_max, ci, cr


def triangular_from_ratio(value, uncertainty=0.20):
    value = float(value)
    if value <= 0:
        raise ValueError('Pairwise comparison values must be positive.')
    return np.array([
        value * (1 - uncertainty),
        value,
        value * (1 + uncertainty),
    ], dtype=float)


def fuzzy_pairwise_from_crisp(pairwise, uncertainty=0.20):
    matrix = np.asarray(pairwise, dtype=float)
    n = matrix.shape[0]
    fuzzy_matrix = np.zeros((n, n, 3), dtype=float)
    for i in range(n):
        for j in range(n):
            if i == j:
                fuzzy_matrix[i, j] = [1.0, 1.0, 1.0]
            elif i < j:
                fuzzy_matrix[i, j] = triangular_from_ratio(matrix[i, j], uncertainty)
                l, m, u = fuzzy_matrix[i, j]
                fuzzy_matrix[j, i] = [1 / u, 1 / m, 1 / l]
    return fuzzy_matrix


def fuzzy_weights_from_pairwise(fuzzy_matrix):
    # Buckley-style fuzzy geometric mean followed by fuzzy normalization.
    n = fuzzy_matrix.shape[0]
    geometric_means = np.zeros((n, 3), dtype=float)
    for i in range(n):
        row_product = np.prod(fuzzy_matrix[i], axis=0)
        geometric_means[i] = row_product ** (1 / n)

    sum_l = geometric_means[:, 0].sum()
    sum_m = geometric_means[:, 1].sum()
    sum_u = geometric_means[:, 2].sum()
    fuzzy_weights = np.column_stack([
        geometric_means[:, 0] / sum_u,
        geometric_means[:, 1] / sum_m,
        geometric_means[:, 2] / sum_l,
    ])
    defuzzified = fuzzy_weights.mean(axis=1)
    defuzzified = defuzzified / defuzzified.sum()
    return fuzzy_weights, defuzzified


def score_to_triangular(score, spread=0.08):
    score = float(np.clip(score, 0, 1))
    return np.array([
        max(0.0, score - spread),
        score,
        min(1.0, score + spread),
    ])


def fuzzy_ahp_scores(profile, pairwise_matrix=None, uncertainty=0.20, eval_spread=0.08):
    pairwise = np.asarray(pairwise_matrix if pairwise_matrix is not None else EXPERT_PAIRWISE_MATRIX, dtype=float)
    crisp_weights, lambda_max, ci, cr = ahp_weights_from_pairwise(pairwise)
    fuzzy_pairwise = fuzzy_pairwise_from_crisp(pairwise, uncertainty=uncertainty)
    fuzzy_weights, defuzz_weights = fuzzy_weights_from_pairwise(fuzzy_pairwise)

    criteria_df = criteria_scores_for_profile(profile)
    criteria_cols = [
        'content_criterion', 'type_criterion', 'location_criterion',
        'distance_criterion', 'quality_criterion'
    ]
    x = criteria_df[criteria_cols].to_numpy(dtype=float)

    fuzzy_eval = np.zeros((len(criteria_df), len(criteria_cols), 3), dtype=float)
    for j in range(len(criteria_cols)):
        fuzzy_eval[:, j, :] = np.vstack([score_to_triangular(value, eval_spread) for value in x[:, j]])

    # H = A x W in triangular fuzzy form.
    fuzzy_h = np.zeros((len(criteria_df), 3), dtype=float)
    for j in range(len(criteria_cols)):
        fuzzy_h += fuzzy_eval[:, j, :] * fuzzy_weights[j]

    # Defuzzification by centroid, then normalization for ranking.
    fuzzy_ahp_score = fuzzy_h.mean(axis=1)
    fuzzy_ahp_norm = minmax(pd.Series(fuzzy_ahp_score, index=criteria_df.index)).to_numpy()

    FUZZY_AHP_INFO.clear()
    FUZZY_AHP_INFO.update({
        'criteria': FUZZY_AHP_CRITERIA,
        'pairwise_matrix': pairwise,
        'crisp_weights': crisp_weights,
        'fuzzy_weights': fuzzy_weights,
        'defuzzified_weights': defuzz_weights,
        'lambda_max': lambda_max,
        'ci': ci,
        'cr': cr,
        'cr_accepted': cr <= 0.10,
    })

    return criteria_df.join(pd.DataFrame({
        'fuzzy_ahp_score': fuzzy_ahp_score,
        'fuzzy_ahp_norm': fuzzy_ahp_norm,
        'final_score': fuzzy_ahp_norm,
    }, index=df.index))


fuzzy_ahp_scores({})
weights_table = pd.DataFrame({
    'criterion': FUZZY_AHP_INFO['criteria'],
    'ahp_weight_from_pairwise': FUZZY_AHP_INFO['crisp_weights'],
    'fuzzy_weight_l': FUZZY_AHP_INFO['fuzzy_weights'][:, 0],
    'fuzzy_weight_m': FUZZY_AHP_INFO['fuzzy_weights'][:, 1],
    'fuzzy_weight_u': FUZZY_AHP_INFO['fuzzy_weights'][:, 2],
    'defuzzified_weight': FUZZY_AHP_INFO['defuzzified_weights'],
})
print('AHP Consistency Ratio:', round(FUZZY_AHP_INFO['cr'], 6), '| accepted:', FUZZY_AHP_INFO['cr_accepted'])
display(weights_table)

## 8. Recommendation Function

`recommend_pois()` ranks POIs with Fuzzy AHP only. The final ranking is based on normalized defuzzified Fuzzy AHP scores.

In [ ]:
def recommend_pois(profile=None, top_k=20, exclude_names=None, min_final_score=0.0, pairwise_matrix=None):
    profile = profile or {}
    exclude_names = {normalize_text(name) for name in (exclude_names or [])}

    result = df.join(fuzzy_ahp_scores(profile, pairwise_matrix=pairwise_matrix))

    if exclude_names:
        result = result[~result['name_key'].isin(exclude_names)]
    result = result[result['final_score'] >= min_final_score]

    output_cols = [
        'Tên địa điểm', 'location', 'poi_type', 'rating_filled', 'review_count',
        'latitude', 'longitude', 'distance_km', 'keywords_clean', 'open_hours_clean',
        'maps_url', 'content_criterion', 'type_criterion', 'location_criterion',
        'distance_criterion', 'quality_criterion', 'fuzzy_ahp_score',
        'fuzzy_ahp_norm', 'final_score'
    ]
    ranked = result.sort_values(['final_score', 'quality_criterion'], ascending=False)
    ranked = ranked[output_cols].head(top_k).copy()
    ranked['distance_km'] = ranked['distance_km'].round(1)
    for col in [
        'content_criterion', 'type_criterion', 'location_criterion',
        'distance_criterion', 'quality_criterion', 'fuzzy_ahp_score',
        'fuzzy_ahp_norm', 'final_score'
    ]:
        ranked[col] = ranked[col].round(4)
    return ranked


def fuzzy_search_poi(query, top_k=10):
    query_norm = normalize_text(query)
    if not query_norm:
        return pd.DataFrame()

    result = df.copy()
    result['fuzzy_name_score'] = result['Tên địa điểm'].map(lambda value: fuzzy_text_score(query_norm, value))
    result['fuzzy_full_text_score'] = result['search_text_norm'].map(lambda value: fuzzy_text_score(query_norm, value))
    result['fuzzy_search_score'] = np.maximum(result['fuzzy_name_score'], result['fuzzy_full_text_score'])
    cols = ['Tên địa điểm', 'location', 'poi_type', 'rating_filled', 'review_count', 'maps_url', 'fuzzy_name_score', 'fuzzy_full_text_score', 'fuzzy_search_score']
    return result.sort_values('fuzzy_search_score', ascending=False)[cols].head(top_k)

## 9. Example Recommendations

Change the profile below to recommend by travel style, place type, location, or current coordinates.

In [ ]:
# Example: cultural + scenic destinations near Nha Trang / Khanh Hoa.
user_profile = {
    'query': 'di tích văn hóa chụp ảnh ngắm cảnh',
    'keywords': ['văn hóa', 'di tích', 'chụp ảnh', 'ngắm cảnh'],
    'preferred_types': ['Tourist attraction', 'Historical landmark', 'Museum', 'Temple', 'Beach'],
    'preferred_locations': ['Khánh Hòa', 'Nha Trang'],
    'current_lat': 12.2388,
    'current_lng': 109.1967,
    'max_distance_km': 120,
}

recommendations = recommend_pois(user_profile, top_k=15)
display(recommendations)

In [ ]:
# Example: fuzzy search handles misspellings and partial user input.
display(fuzzy_search_poi('thap ba ponaga', top_k=10))
display(fuzzy_search_poi('vin wonder nha trang', top_k=10))

In [ ]:
# Example: beach/leisure recommendation around Da Nang.
da_nang_profile = {
    'query': 'biển vui chơi nghỉ dưỡng chụp ảnh',
    'keywords': ['biển', 'vui chơi', 'nghỉ dưỡng', 'chụp ảnh'],
    'preferred_types': ['Beach', 'Tourist attraction', 'Park'],
    'preferred_locations': ['Đà Nẵng', 'Hội An', 'Quảng Nam'],
    'current_lat': 16.0471,
    'current_lng': 108.2062,
    'max_distance_km': 80,
}

display(recommend_pois(da_nang_profile, top_k=15))

## 10. Evaluation, Metrics, and Baseline Models

There is no explicit user-click or user-rating ground truth in the workbook, so evaluation uses weak/pseudo relevance labels generated from held-out user profiles. A POI is considered relevant for a profile when it scores highly on a balanced oracle made from separated content, type, location, distance, and quality criteria.

Baseline models compared:

- `popularity_baseline`: rank by quality only
- `distance_baseline`: rank by distance only
- `content_baseline`: rank by content TF-IDF only
- `type_location_baseline`: rank by type and location match only
- `equal_weight_baseline`: rank by simple equal-weight criteria average
- `crisp_ahp_weighted_baseline`: rank by crisp AHP weighted criteria without fuzzy weights
- `fuzzy_ahp`: main model

Metrics:

- Precision@K
- Recall@K
- MAP@K
- MRR@K
- NDCG@K
- type diversity@K
- catalog coverage@K

In [ ]:
EVAL_K = 10

EVAL_PROFILES = [
    {
        'name': 'culture_nha_trang',
        'query': 'culture heritage museum temple viewpoint photo',
        'keywords': ['culture', 'heritage', 'museum', 'temple', 'viewpoint'],
        'preferred_types': ['Tourist attraction', 'Historical landmark', 'Museum', 'Temple', 'Buddhist temple'],
        'preferred_locations': ['Khanh Hoa', 'Nha Trang'],
        'current_lat': 12.2388,
        'current_lng': 109.1967,
        'max_distance_km': 120,
    },
    {
        'name': 'beach_da_nang',
        'query': 'beach leisure photo scenic park',
        'keywords': ['beach', 'leisure', 'photo', 'scenic'],
        'preferred_types': ['Beach', 'Tourist attraction', 'Park', 'Scenic spot'],
        'preferred_locations': ['Da Nang', 'Hoi An', 'Quang Nam'],
        'current_lat': 16.0471,
        'current_lng': 108.2062,
        'max_distance_km': 90,
    },
    {
        'name': 'spiritual_hanoi',
        'query': 'temple pagoda church spiritual culture',
        'keywords': ['temple', 'pagoda', 'church', 'spiritual', 'culture'],
        'preferred_types': ['Buddhist temple', 'Catholic church', 'Place of worship', 'Pagoda', 'Shrine'],
        'preferred_locations': ['Ha Noi', 'Hanoi'],
        'current_lat': 21.0278,
        'current_lng': 105.8342,
        'max_distance_km': 80,
    },
    {
        'name': 'nature_mountain_north',
        'query': 'mountain viewpoint waterfall lake national park',
        'keywords': ['mountain', 'viewpoint', 'waterfall', 'lake', 'national park'],
        'preferred_types': ['Scenic spot', 'Viewpoint', 'National park', 'Lake', 'Tourist attraction'],
        'preferred_locations': ['Lao Cai', 'Ha Giang', 'Ninh Binh'],
        'current_lat': 22.3364,
        'current_lng': 103.8438,
        'max_distance_km': 250,
    },
]


def pseudo_relevance(profile):
    criteria = criteria_scores_for_profile(profile)
    oracle = (
        0.25 * criteria['content_criterion'] +
        0.15 * criteria['type_criterion'] +
        0.15 * criteria['location_criterion'] +
        0.20 * criteria['distance_criterion'] +
        0.25 * criteria['quality_criterion']
    )
    threshold = oracle.quantile(0.95)
    relevance = (oracle >= threshold).astype(int)
    if relevance.sum() < EVAL_K:
        relevance.loc[oracle.sort_values(ascending=False).head(EVAL_K).index] = 1
    return oracle, relevance


def precision_at_k(relevance_ranked, k):
    values = np.asarray(relevance_ranked[:k], dtype=float)
    return values.mean() if len(values) else 0.0


def recall_at_k(relevance_ranked, total_relevant, k):
    if total_relevant == 0:
        return 0.0
    return float(np.asarray(relevance_ranked[:k]).sum() / total_relevant)


def average_precision_at_k(relevance_ranked, k):
    values = np.asarray(relevance_ranked[:k], dtype=float)
    hits = 0
    precisions = []
    for idx, rel in enumerate(values, start=1):
        if rel > 0:
            hits += 1
            precisions.append(hits / idx)
    return float(np.mean(precisions)) if precisions else 0.0


def reciprocal_rank_at_k(relevance_ranked, k):
    for idx, rel in enumerate(relevance_ranked[:k], start=1):
        if rel > 0:
            return 1 / idx
    return 0.0


def ndcg_at_k(relevance_ranked, relevance_scores_ranked, k):
    gains = np.asarray(relevance_scores_ranked[:k], dtype=float) * np.asarray(relevance_ranked[:k], dtype=float)
    discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
    dcg = float(np.sum(gains * discounts))
    ideal_gains = np.sort(gains)[::-1]
    idcg = float(np.sum(ideal_gains * discounts))
    return dcg / idcg if idcg > 0 else 0.0


def ranking_diversity(indices):
    subset = df.loc[indices]
    type_diversity = subset['poi_type'].nunique() / max(1, len(subset))
    return float(type_diversity)


def crisp_ahp_weighted_scores(profile):
    criteria = criteria_scores_for_profile(profile)
    weights, _, _, _ = ahp_weights_from_pairwise(EXPERT_PAIRWISE_MATRIX)
    cols = ['content_criterion', 'type_criterion', 'location_criterion', 'distance_criterion', 'quality_criterion']
    return pd.Series(criteria[cols].to_numpy(dtype=float) @ weights, index=df.index)


def model_scores(profile):
    criteria = criteria_scores_for_profile(profile)
    main = fuzzy_ahp_scores(profile)['final_score']
    return {
        'popularity_baseline': criteria['quality_criterion'],
        'distance_baseline': criteria['distance_criterion'],
        'content_baseline': criteria['content_criterion'],
        'type_location_baseline': 0.50 * criteria['type_criterion'] + 0.50 * criteria['location_criterion'],
        'equal_weight_baseline': criteria[[
            'content_criterion', 'type_criterion', 'location_criterion',
            'distance_criterion', 'quality_criterion'
        ]].mean(axis=1),
        'crisp_ahp_weighted_baseline': crisp_ahp_weighted_scores(profile),
        'fuzzy_ahp': main,
    }


def evaluate_models(profiles, k=10):
    rows = []
    coverage_tracker = {}
    for profile in profiles:
        oracle, relevance = pseudo_relevance(profile)
        total_relevant = int(relevance.sum())
        scores_by_model = model_scores(profile)
        for model_name, scores in scores_by_model.items():
            ranked_indices = scores.sort_values(ascending=False).head(k).index
            rel_ranked = relevance.loc[ranked_indices].to_numpy()
            oracle_ranked = oracle.loc[ranked_indices].to_numpy()
            coverage_tracker.setdefault(model_name, set()).update(ranked_indices.tolist())
            rows.append({
                'profile': profile['name'],
                'model': model_name,
                f'precision@{k}': precision_at_k(rel_ranked, k),
                f'recall@{k}': recall_at_k(rel_ranked, total_relevant, k),
                f'map@{k}': average_precision_at_k(rel_ranked, k),
                f'mrr@{k}': reciprocal_rank_at_k(rel_ranked, k),
                f'ndcg@{k}': ndcg_at_k(rel_ranked, oracle_ranked, k),
                f'type_diversity@{k}': ranking_diversity(ranked_indices),
            })
    metrics = pd.DataFrame(rows)
    summary = metrics.groupby('model').mean(numeric_only=True).reset_index()
    summary[f'catalog_coverage@{k}'] = summary['model'].map(lambda model: len(coverage_tracker.get(model, set())) / len(df))
    return metrics, summary.sort_values(f'ndcg@{k}', ascending=False)


evaluation_by_profile, evaluation_summary = evaluate_models(EVAL_PROFILES, EVAL_K)
display(evaluation_by_profile)
display(evaluation_summary)

## 11. Save Cleaned Data and Sample Recommendation Output

The cleaned file is useful for later modeling, dashboards, or a web/app recommender. It keeps original user-facing columns plus normalized recommendation features.

In [ ]:
export_cols = [
    'STT', 'Tên địa điểm', 'location', 'description', 'rating_filled', 'review_count',
    'image_url', 'keywords_clean', 'poi_type', 'latitude', 'longitude', 'open_hours_clean',
    'quality_score', 'content_text_norm', 'search_text_norm', 'maps_url'
]
clean_export = df[export_cols].copy()
clean_export = clean_export.rename(columns={
    'location': 'Vị trí_clean',
    'description': 'Mô tả_clean',
    'rating_filled': 'rating_clean',
    'review_count': 'review_count_clean',
    'image_url': 'Ảnh_clean',
    'keywords_clean': 'Từ Khóa_clean',
    'poi_type': 'destination_type_clean',
    'latitude': 'lat_clean',
    'longitude': 'lng_clean',
})

clean_export.to_excel(CLEAN_OUTPUT_PATH, index=False)
recommendations.to_excel(SAMPLE_RECOMMENDATIONS_PATH, index=False)
with pd.ExcelWriter(EVALUATION_OUTPUT_PATH) as writer:
    evaluation_summary.to_excel(writer, sheet_name='summary', index=False)
    evaluation_by_profile.to_excel(writer, sheet_name='by_profile', index=False)
    weights_table.to_excel(writer, sheet_name='fuzzy_ahp_weights', index=False)

print('Saved cleaned recommendation data:', CLEAN_OUTPUT_PATH)
print('Saved sample recommendations:', SAMPLE_RECOMMENDATIONS_PATH)
print('Saved evaluation metrics:', EVALUATION_OUTPUT_PATH)
print('Clean export shape:', clean_export.shape)

## 12. Notes for Tuning

- Edit `EXPERT_PAIRWISE_MATRIX` to reflect user/expert pairwise judgments.
- Do not edit final weights directly; AHP computes weights from the pairwise matrix.
- Check `CR <= 0.10`; if not, the expert judgments are too inconsistent.
- Keep content, type, and location as separate criteria to avoid double-counting.
- Use `preferred_locations` for province/city-style matching without hard filters.
- Use `exclude_names` in `recommend_pois()` to remove places a user has already visited.
- If `rapidfuzz` is installed, fuzzy matching will be faster and more accurate; otherwise the notebook falls back to `difflib`.